# Recalce ML Pipeline — Monolithic Notebook

**Kaggle-ready, single-file implementation** of the complete pipeline:
data generation → feature engineering → model training → evaluation.

All functions are defined inline — no external module imports required.
Run every cell top-to-bottom.

---
**Dataset required on Kaggle:** [PaySim1](https://www.kaggle.com/datasets/ntnu-testimon/paysim1)

## 1. Import Dependencies

In [47]:
import json
import os
import random
import warnings
from dataclasses import asdict, dataclass
from datetime import datetime, timedelta
from decimal import Decimal, ROUND_HALF_UP
from math import ceil, sqrt
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    precision_recall_curve,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
print("Dependencies loaded.")

Dependencies loaded.


## 2. Configuration

Edit the paths and row counts here. PaySim CSV is auto-detected under `/kaggle/input`.

In [48]:
# ── Row counts ────────────────────────────────────────────────────────────────
N_TRAIN_ROWS = 50000
N_TEST_ROWS  = 10000
RANDOM_STATE = 42

# 1. Hardcode the Paths using pathlib
PAYSIM_PATH = Path("/kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv")
OUT_DIR = Path("/kaggle/working/ml/data")
MODELS_DIR = Path("/kaggle/working/ml/models")
DATA_DIR = OUT_DIR  # alias used by train / evaluate functions

# 2. Create directories if they don't exist
# exist_ok=True ensures that if the folder is already there, it won't crash, 
# and any subsequent CSV or PKL file saves will smoothly overwrite the old files.
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PaySim path : {PAYSIM_PATH}")
print(f"Output dir  : {OUT_DIR}")
print(f"Models dir  : {MODELS_DIR}")

PaySim path : /kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv
Output dir  : /kaggle/working/ml/data
Models dir  : /kaggle/working/ml/models


## 3. Data Generation

Generates synthetic reconciliation datasets from PaySim.
Produces per split: `<split>_ledger.csv`, `<split>_bank.csv`, `<split>_ground_truth.csv`.

In [49]:
# ── Constants ─────────────────────────────────────────────────────────────────
OPERATIONAL_BUCKETS = [
    ("exact",      0.55),
    ("date_shift", 0.12),
    ("fee_adj",    0.08),
    ("duplicate",  0.02),
    ("missing",    0.03),
    ("group",      0.15),
    ("ambiguous",  0.03),
]

DATABASE_RULE_BUCKETS = {"duplicate", "ambiguous"}
ANOMALY_BUCKETS       = {"anomaly"}

SETTLEMENT_WINDOW_DAYS = 3
FEE_MIN = 0.01
FEE_MAX = 0.03


# ── Helpers ───────────────────────────────────────────────────────────────────
def to_cents(amount_float: float) -> int:
    """Convert float to integer cents — never store float in output."""
    return int(Decimal(str(amount_float)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP) * 100)


def cents_to_decimal_str(cents: int) -> str:
    """Integer cents → '1999' → '19.99' string, safe for Decimal() parsing."""
    return str(Decimal(cents) / 100)


def make_txn_id(n: int) -> str:
    return f"TXN{n:07d}"


def make_batch_id(n: int) -> str:
    return f"BATCH{n:07d}"


def base_date_for_split(split: str) -> datetime:
    """Keep train and test periods independent and relative to run time."""
    days_back = 90 if split == "train" else 30
    return datetime.now() - timedelta(days=days_back)


def step_to_timestamp(step: int, base_date: datetime, jitter_minutes: int = 0) -> datetime:
    """PaySim step is simulation-hour (1-744). Map to real datetime."""
    return base_date + timedelta(hours=int(step), minutes=jitter_minutes)


def anomaly_counts(n_rows: int) -> tuple:
    """Create enough positives for both model populations without a fixed rate."""
    per_model = max(1, int(np.ceil(np.sqrt(n_rows))))
    matched   = min(per_model, n_rows)
    unmatched = min(per_model, max(0, n_rows - matched))
    return matched, unmatched


def assign_buckets(n: int, rng: random.Random) -> list:
    """Assign operational buckets, then reserve dynamic anomalies per model."""
    names   = [bucket for bucket, _ in OPERATIONAL_BUCKETS]
    weights = [weight for _, weight in OPERATIONAL_BUCKETS]
    buckets = rng.choices(names, weights=weights, k=n)

    matched_count, unmatched_count = anomaly_counts(n)
    anomaly_indices = rng.sample(range(n), k=matched_count + unmatched_count)
    for index in anomaly_indices[:matched_count]:
        buckets[index] = "anomaly_matched"
    for index in anomaly_indices[matched_count:]:
        buckets[index] = "anomaly_unmatched"
    return buckets


# ── Load PaySim ───────────────────────────────────────────────────────────────
def load_paysim(n_rows: int, rng: random.Random, base_date: datetime) -> tuple:
    print(f"[generate] Loading PaySim from {PAYSIM_PATH} ...")
    df = pd.read_csv(
        PAYSIM_PATH,
        usecols=["step", "type", "amount", "nameDest"],
        dtype={"step": int, "type": str, "amount": float, "nameDest": str},
    )
    df = df[df["type"].isin(["PAYMENT", "TRANSFER"])].copy()
    df = df.sample(n=min(n_rows, len(df)), random_state=rng.randint(0, 99999)).reset_index(drop=True)

    dynamic_merchant_count = max(10, int(np.sqrt(n_rows)))
    unique_dests = df["nameDest"].unique()
    dest_to_merchant = {
        d: f"MER{(i % dynamic_merchant_count):03d}"
        for i, d in enumerate(unique_dests)
    }
    df["merchant_id"] = df["nameDest"].map(dest_to_merchant)

    df_sorted  = df.sort_values(["nameDest", "step"])
    step_diffs = df_sorted.groupby("nameDest")["step"].diff().dropna()
    step_diffs = step_diffs[step_diffs > 0]
    required_diffs = max(10, int(np.ceil(np.sqrt(len(df)))))
    if len(step_diffs) >= required_diffs:
        settlement_window_days = max(1, int(np.ceil(np.percentile(step_diffs, 75) / 24)))
    else:
        settlement_window_days = SETTLEMENT_WINDOW_DAYS

    df["amount_cents"] = df["amount"].apply(to_cents)
    df["timestamp"] = df.apply(
        lambda r: step_to_timestamp(r["step"], base_date, jitter_minutes=rng.randint(0, 59)),
        axis=1,
    )
    df["transaction_id"] = [make_txn_id(i + 1) for i in range(len(df))]

    return (
        df[["transaction_id", "amount_cents", "timestamp", "merchant_id"]].copy(),
        settlement_window_days,
    )


# ── Row writers ───────────────────────────────────────────────────────────────
def _upper_fence(values: np.ndarray) -> float:
    """Return the data-derived outer Tukey fence."""
    q1, q3 = np.percentile(values, [25, 75])
    return float(q3 + 3 * (q3 - q1))


def matched_anomaly_profile(n_rows: int, settlement_window_days: int, rng: random.Random) -> tuple:
    """Derive abnormal fee and delay levels from simulated normal behavior."""
    profile_size     = max(4, int(np.ceil(np.sqrt(n_rows))))
    normal_fee_rates = np.array([rng.uniform(FEE_MIN, FEE_MAX) for _ in range(profile_size)])
    normal_delays    = np.array([rng.randint(0, settlement_window_days) for _ in range(profile_size)])

    fee_rate   = min(0.95, max(_upper_fence(normal_fee_rates), normal_fee_rates.max()))
    delay_days = max(
        int(np.ceil(_upper_fence(normal_delays))),
        int(normal_delays.max()) + 1,
    )
    return fee_rate, delay_days


def unmatched_anomaly_amounts(df: pd.DataFrame) -> dict:
    """Build merchant-specific outlier amounts from normal PaySim histories."""
    outlier_amounts = {}
    for merchant, history in df.groupby("merchant_id"):
        amounts     = history["amount_cents"].to_numpy(dtype=float)
        median      = float(np.median(amounts))
        upper_fence = _upper_fence(amounts)
        dynamic_gap = max(float(amounts.max()) - median, upper_fence - median, 1.0)
        outlier_amounts[merchant] = int(np.ceil(max(float(amounts.max()) + dynamic_gap, upper_fence)))
    return outlier_amounts


def write_exact(row, ledger_rows, bank_rows, gt_rows):
    bank_rows.append({
        "bank_reference_id": row["transaction_id"],
        "deposit_amount":    row["amount_cents"],
        "settlement_date":   row["timestamp"].date(),
    })
    gt_rows.append({**row, "bucket": "exact", "group_id": None, "expected_status": "MATCHED"})


def write_date_shift(row, ledger_rows, bank_rows, gt_rows, rng, settlement_window_days: int):
    shift = rng.randint(1, settlement_window_days)
    bank_rows.append({
        "bank_reference_id": row["transaction_id"],
        "deposit_amount":    row["amount_cents"],
        "settlement_date":   (row["timestamp"] + timedelta(days=shift)).date(),
    })
    gt_rows.append({**row, "bucket": "date_shift", "group_id": None, "expected_status": "MATCHED"})


def write_fee_adj(row, ledger_rows, bank_rows, gt_rows, rng, settlement_window_days: int):
    fee_rate  = Decimal(str(round(rng.uniform(FEE_MIN, FEE_MAX), 4)))
    fee_cents = int((Decimal(row["amount_cents"]) * fee_rate).quantize(Decimal("1"), rounding=ROUND_HALF_UP))
    deposit   = row["amount_cents"] - fee_cents
    shift     = rng.randint(0, settlement_window_days)
    bank_rows.append({
        "bank_reference_id": row["transaction_id"],
        "deposit_amount":    deposit,
        "settlement_date":   (row["timestamp"] + timedelta(days=shift)).date(),
    })
    gt_rows.append({**row, "bucket": "fee_adj", "group_id": None, "expected_status": "MATCHED"})


def write_duplicate(row, ledger_rows, bank_rows, gt_rows, rng):
    shift = rng.randint(0, 1)
    for _ in range(2):
        bank_rows.append({
            "bank_reference_id": row["transaction_id"],
            "deposit_amount":    row["amount_cents"],
            "settlement_date":   (row["timestamp"] + timedelta(days=shift)).date(),
        })
    gt_rows.append({**row, "bucket": "duplicate", "group_id": None, "expected_status": "MATCHED"})


def write_missing(row, ledger_rows, bank_rows, gt_rows):
    gt_rows.append({**row, "bucket": "missing", "group_id": None, "expected_status": "UNRECONCILED"})


def write_matched_anomaly(row, ledger_rows, bank_rows, gt_rows, fee_rate: float, delay_days: int, rng: random.Random):
    """Create a matched anomaly visible through fee_ratio and settle_delay."""
    row_fee = rng.uniform(fee_rate, min(0.95, fee_rate * 1.5))
    row_delay = rng.randint(delay_days, delay_days * 2);
    unusual_cents = max(1, int(row["amount_cents"] * (1 - row_fee)))
    bank_rows.append({
        "bank_reference_id": row["transaction_id"],
        "deposit_amount":    unusual_cents,
        "settlement_date":   (row["timestamp"] + timedelta(days=row_delay)).date(),
    })
    gt_rows.append({
        **row,
        "bucket": "anomaly",
        "anomaly_variant": "matched_fee_delay",
        "group_id": None,
        "expected_status": "MATCHED",
    })


def write_unmatched_anomaly(row, ledger_rows, bank_rows, gt_rows):
    """Create an unmatched anomaly visible through merchant amount z-score."""
    gt_rows.append({
        **row,
        "bucket": "anomaly",
        "anomaly_variant": "unmatched_amount",
        "group_id": None,
        "expected_status": "UNRECONCILED",
    })


# ── Group builders ────────────────────────────────────────────────────────────
def build_groups(df: pd.DataFrame, rng: random.Random, group_id_start: int):
    """2-5 transactions from the same merchant → 1 bank row."""
    results     = []
    gid         = group_id_start
    by_merchant = {m: grp.to_dict("records") for m, grp in df.groupby("merchant_id")}

    for _, pool in by_merchant.items():
        if len(pool) < 2:
            continue
        rng.shuffle(pool)
        i = 0
        while i + 2 <= len(pool):
            size    = rng.randint(2, min(5, len(pool) - i))
            members = pool[i: i + size]
            total_cents = sum(r["amount_cents"] for r in members)
            batch_ref   = make_batch_id(gid)
            max_ts      = max(r["timestamp"] for r in members)
            settle_date = (max_ts + timedelta(days=rng.randint(0, 2))).date()
            bank_row = {
                "bank_reference_id": batch_ref,
                "deposit_amount":    total_cents,
                "settlement_date":   settle_date,
            }
            gt = [
                {**r, "bucket": "group", "group_id": batch_ref, "expected_status": "MATCHED"}
                for r in members
            ]
            results.append((members, bank_row, gt))
            gid += 1
            i   += size

    return results


def build_ambiguous_groups(df: pd.DataFrame, rng: random.Random, group_id_start: int, count: int):
    """Deliberately construct pairs where two disjoint subsets sum to the same target."""
    results = []
    gid     = group_id_start
    rows    = df.to_dict("records")
    rng.shuffle(rows)

    attempts = 0
    found    = 0
    idx      = 0

    while found < count and idx + 6 <= len(rows) and attempts < count * 20:
        attempts += 1
        chunk = rows[idx: idx + 6]
        a, b, c, d, e, f = chunk
        target   = a["amount_cents"] + b["amount_cents"] + c["amount_cents"]
        f_needed = target - d["amount_cents"] - e["amount_cents"]
        if f_needed <= 0:
            idx += 3
            continue

        f = dict(f)
        f["amount_cents"] = f_needed

        batch_ref   = make_batch_id(gid)
        settle_date = (max(r["timestamp"] for r in chunk) + timedelta(days=1)).date()
        bank_row    = {
            "bank_reference_id": batch_ref,
            "deposit_amount":    target,
            "settlement_date":   settle_date,
        }
        gt = [
            {**r, "bucket": "ambiguous", "group_id": batch_ref, "expected_status": "UNDER_REVIEW"}
            for r in [a, b, c, d, e, f]
        ]
        results.append(([a, b, c, d, e, f], bank_row, gt))
        gid   += 1
        found += 1
        idx   += 6

    return results


# ── Main generate function ────────────────────────────────────────────────────
def generate(split: str, seed: int, n_rows: int):
    """Generate train or test CSVs from PaySim.

    Args:
        split:  'train' or 'test'
        seed:   random seed for reproducibility
        n_rows: number of ledger rows to generate
    """
    rng = random.Random(seed)
    np.random.seed(seed)

    base_date = base_date_for_split(split)
    paysim, settlement_window_days = load_paysim(n_rows, rng, base_date)

    buckets = assign_buckets(len(paysim), rng)
    paysim["bucket"] = buckets
    matched_fee_rate, matched_delay_days = matched_anomaly_profile(
        len(paysim), settlement_window_days, rng
    )
    unmatched_amount_by_merchant = unmatched_anomaly_amounts(paysim)

    ledger_rows = []
    bank_rows   = []
    gt_rows     = []

    group_pool_indices     = []
    ambiguous_pool_indices = []

    for i, (_, row) in enumerate(paysim.iterrows()):
        r      = row.to_dict()
        bucket = r["bucket"]

        if bucket == "anomaly_unmatched":
            r["amount_cents"] = unmatched_amount_by_merchant[r["merchant_id"]]

        ledger_rows.append({
            "transaction_id": r["transaction_id"],
            "amount":         cents_to_decimal_str(r["amount_cents"]),
            "timestamp":      r["timestamp"].isoformat(),
            "merchant_id":    r["merchant_id"],
        })

        if bucket == "exact":
            write_exact(r, ledger_rows, bank_rows, gt_rows)
        elif bucket == "date_shift":
            write_date_shift(r, ledger_rows, bank_rows, gt_rows, rng, settlement_window_days)
        elif bucket == "fee_adj":
            write_fee_adj(r, ledger_rows, bank_rows, gt_rows, rng, settlement_window_days)
        elif bucket == "duplicate":
            write_duplicate(r, ledger_rows, bank_rows, gt_rows, rng)
        elif bucket == "missing":
            write_missing(r, ledger_rows, bank_rows, gt_rows)
        elif bucket == "anomaly_matched":
            write_matched_anomaly(r, ledger_rows, bank_rows, gt_rows, matched_fee_rate, matched_delay_days, rng)
        elif bucket == "anomaly_unmatched":
            write_unmatched_anomaly(r, ledger_rows, bank_rows, gt_rows)
        elif bucket == "group":
            group_pool_indices.append(i)
            gt_rows.append({**r, "bucket": "group", "group_id": None, "expected_status": "MATCHED"})
        elif bucket == "ambiguous":
            ambiguous_pool_indices.append(i)
            gt_rows.append({**r, "bucket": "ambiguous", "group_id": None, "expected_status": "UNDER_REVIEW"})

    group_df = paysim.iloc[group_pool_indices].copy()
    groups   = build_groups(group_df, rng, group_id_start=1)
    for members, bank_row, member_gt in groups:
        bank_rows.append(bank_row)
        for gt_row in gt_rows:
            if gt_row.get("bucket") == "group" and any(
                gt_row["transaction_id"] == m["transaction_id"] for m in members
            ):
                gt_row["group_id"] = bank_row["bank_reference_id"]

    ambiguous_df = paysim.iloc[ambiguous_pool_indices].copy() if ambiguous_pool_indices else pd.DataFrame()
    n_ambiguous  = max(1, len(ambiguous_pool_indices) // 6)
    if not ambiguous_df.empty:
        ambiguous_groups = build_ambiguous_groups(ambiguous_df, rng, group_id_start=10000, count=n_ambiguous)
        for members, bank_row, member_gt in ambiguous_groups:
            bank_rows.append(bank_row)

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    ledger_df = pd.DataFrame(ledger_rows)
    bank_df   = pd.DataFrame(bank_rows)
    gt_df     = pd.DataFrame(gt_rows)
    gt_df["ml_eligible"] = ~gt_df["bucket"].isin(DATABASE_RULE_BUCKETS)

    ledger_path = OUT_DIR / f"{split}_ledger.csv"
    bank_path   = OUT_DIR / f"{split}_bank.csv"
    gt_path     = OUT_DIR / f"{split}_ground_truth.csv"

    ledger_df.to_csv(ledger_path, index=False)
    bank_df.to_csv(bank_path, index=False)
    gt_df.to_csv(gt_path, index=False)

    print(f"[generate] {split}: {len(ledger_df)} ledger rows, {len(bank_df)} bank rows")
    print(
        "  Dynamic configuration: "
        f"merchants={max(10, int(np.sqrt(n_rows)))}, "
        f"settlement_window={settlement_window_days} days, "
        f"matched_anomaly_fee={matched_fee_rate:.4f}, "
        f"matched_anomaly_delay={matched_delay_days} days"
    )
    print(f"  Bucket breakdown:\n{gt_df['bucket'].value_counts().to_string()}")
    print(f"  Written: {ledger_path}, {bank_path}, {gt_path}")


print("Data generation functions defined.")

Data generation functions defined.


## 4. Feature Engineering

Two feature sets for two separate models (different dimensionality):
- `matched_features()` → `[amount, hour_of_day, day_of_week, merchant_freq, fee_ratio, settle_delay]` (6 features)
- `unmatched_features()` → `[amount, hour_of_day, day_of_week, merchant_freq, amount_zscore]` (5 features)

In [50]:
def _merchant_frequency(df: pd.DataFrame) -> pd.Series:
    """Count of transactions per merchant_id in the batch."""
    freq = df["merchant_id"].value_counts()
    return df["merchant_id"].map(freq).astype(float)


def minimum_transactions_for_zscore(df: pd.DataFrame) -> int:
    """Derive the merchant-history length needed for a stable standard deviation."""
    if df.empty:
        return 3

    columns  = ["merchant_id", "amount"]
    histories = df[columns].copy()
    if "timestamp" in df.columns:
        histories["timestamp"] = pd.to_datetime(df["timestamp"])
        histories = histories.sort_values(["merchant_id", "timestamp"])
    else:
        histories = histories.sort_values(["merchant_id"])

    stability_points = []
    for _, merchant_history in histories.groupby("merchant_id", sort=False):
        values = merchant_history["amount"].astype(float).to_numpy()
        if len(values) < 4:
            continue

        current_std = float(np.std(values[:3], ddof=1))
        for n in range(3, len(values)):
            next_std = float(np.std(values[: n + 1], ddof=1))
            if current_std <= 1e-9:
                relative_change = 0.0 if next_std <= 1e-9 else np.inf
            else:
                relative_change = abs(next_std - current_std) / current_std

            if relative_change < 0.10:
                stability_points.append(n)
                break
            current_std = next_std

    if stability_points:
        return int(np.clip(np.ceil(np.median(stability_points)), 3, 15))

    return int(np.clip(np.ceil(np.sqrt(len(df))), 3, 15))


def _safe_std(values: pd.Series) -> float:
    """Return a finite, non-zero sample standard deviation."""
    std = float(values.astype(float).std(ddof=1))
    return std if np.isfinite(std) and std > 1e-9 else 1e-9


def _amount_zscore_per_merchant(df: pd.DataFrame, min_txns=None) -> pd.Series:
    """Z-score of amount within each merchant's transactions."""
    amounts    = df["amount"].astype(float)
    batch_mean = amounts.mean()
    batch_std  = _safe_std(amounts)
    min_txns   = minimum_transactions_for_zscore(df) if min_txns is None else min_txns

    scores = pd.Series(index=df.index, dtype=float)
    for merchant, grp in df.groupby("merchant_id"):
        if len(grp) >= min_txns:
            m = grp["amount"].astype(float).mean()
            s = _safe_std(grp["amount"])
        else:
            m, s = batch_mean, batch_std
        scores.loc[grp.index] = (grp["amount"].astype(float) - m) / s

    return scores


def matched_features(matched_df: pd.DataFrame) -> np.ndarray:
    """Build feature matrix for matched records.

    Required columns: amount, timestamp, merchant_id, fee_deducted, settlement_delay_days
    Returns shape: (n_samples, 6)
    """
    df = matched_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    features = pd.DataFrame({
        "amount":        df["amount"].astype(float),
        "hour_of_day":   df["timestamp"].dt.hour.astype(float),
        "day_of_week":   df["timestamp"].dt.dayofweek.astype(float),
        "merchant_freq": _merchant_frequency(df),
        "fee_ratio":     df["fee_deducted"].astype(float) / df["amount"].astype(float).clip(lower=1e-9),
        "settle_delay":  df["settlement_delay_days"].astype(float),
    })
    return features.to_numpy()


def unmatched_features(unmatched_df: pd.DataFrame) -> np.ndarray:
    """Build feature matrix for unmatched records.

    Required columns: amount, timestamp, merchant_id
    Returns shape: (n_samples, 5)
    """
    df = unmatched_df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    features = pd.DataFrame({
        "amount":        df["amount"].astype(float),
        "hour_of_day":   df["timestamp"].dt.hour.astype(float),
        "day_of_week":   df["timestamp"].dt.dayofweek.astype(float),
        "merchant_freq": _merchant_frequency(df),
        "amount_zscore": _amount_zscore_per_merchant(df),
    })
    return features.to_numpy()


print("Feature engineering functions defined.")

Feature engineering functions defined.


## 5. Threshold Calibration

IsolationForest is fit **unsupervised**. Labels are used only to select the score cutoff
that maximises precision while meeting a data-derived recall safety floor.

In [51]:
@dataclass(frozen=True)
class CalibrationResult:
    """A score cutoff and the validation metrics that selected it."""
    score_threshold:      float
    recall_floor:         float
    precision:            float
    recall:               float
    f1:                   float
    alert_rate:           float
    validation_anomalies: int
    validation_rows:      int

    def to_dict(self):
        return asdict(self)


def validation_fraction(labels: np.ndarray) -> float:
    """Choose a validation share that contains roughly sqrt(positive) anomalies."""
    positive_count = int(np.asarray(labels, dtype=int).sum())
    if positive_count < 2:
        raise ValueError(
            "At least two labelled anomalies are required for validation calibration."
        )
    validation_positives = int(ceil(sqrt(positive_count)))
    return min(0.5, validation_positives / positive_count)


def recall_safety_floor(labels: np.ndarray) -> float:
    """Require validation recall high enough to miss at most one anomaly."""
    positive_count = int(np.asarray(labels, dtype=int).sum())
    if positive_count < 2:
        raise ValueError(
            "At least two validation anomalies are required to derive a recall floor."
        )
    return (positive_count - 1) / positive_count


def select_precision_first_threshold(
    labels: np.ndarray,
    anomaly_scores: np.ndarray,
) -> CalibrationResult:
    """Select the most precise score cutoff that satisfies the recall floor."""
    y_true = np.asarray(labels, dtype=int)
    scores = np.asarray(anomaly_scores, dtype=float)
    if len(y_true) != len(scores):
        raise ValueError("Labels and anomaly scores must have the same length.")
    if not np.isfinite(scores).all():
        raise ValueError("Anomaly scores must be finite for threshold calibration.")

    floor = recall_safety_floor(y_true)
    precision_vals, recall_vals, thresholds = precision_recall_curve(y_true, scores)
    candidates = []

    for index, threshold in enumerate(thresholds):
        candidate_precision = float(precision_vals[index])
        candidate_recall    = float(recall_vals[index])
        if candidate_recall < floor:
            continue

        predicted  = scores >= threshold
        alert_rate = float(predicted.mean())
        denominator = candidate_precision + candidate_recall
        f1 = (
            0.0
            if denominator == 0.0
            else 2 * candidate_precision * candidate_recall / denominator
        )
        candidates.append(
            CalibrationResult(
                score_threshold=float(threshold),
                recall_floor=float(floor),
                precision=candidate_precision,
                recall=candidate_recall,
                f1=float(f1),
                alert_rate=alert_rate,
                validation_anomalies=int(y_true.sum()),
                validation_rows=len(y_true),
            )
        )

    if not candidates:
        raise RuntimeError("No validation score threshold satisfied the recall safety floor.")

    return sorted(
        candidates,
        key=lambda result: (
            -result.precision,
            result.alert_rate,
            -result.f1,
            -result.score_threshold,
        ),
    )[0]


def apply_score_threshold(model, score_threshold: float) -> None:
    """Configure an IsolationForest instance so predict() uses this cutoff."""
    model.offset_ = -float(score_threshold)


print("Calibration functions defined.")

Calibration functions defined.


## 6. Training Pipeline

Two IsolationForest models (matched vs. unmatched) fitted on the generated training data.
Calibration selects the score threshold; IsolationForest itself is always **unsupervised**.

In [52]:
def _simple_waterfall(ledger: pd.DataFrame, bank: pd.DataFrame):
    """Simplified in-memory waterfall — produces matched / unmatched splits.

    Returns:
        matched_df   : rows that have a corresponding bank_reference_id
        unmatched_df : rows with no bank match
    """
    bank_idx = set(bank["bank_reference_id"].tolist())

    matched_rows   = []
    unmatched_rows = []

    for _, row in ledger.iterrows():
        txn_id = row["transaction_id"]
        if txn_id in bank_idx:
            bank_row = bank[bank["bank_reference_id"] == txn_id].iloc[0]
            fee   = float(row["amount"]) - float(bank_row["deposit_amount"])
            ts    = pd.to_datetime(row["timestamp"])
            sd    = pd.to_datetime(bank_row["settlement_date"])
            delay = max(0, (sd.date() - ts.date()).days)
            matched_rows.append({
                "transaction_id":        txn_id,
                "amount":                row["amount"],
                "timestamp":             row["timestamp"],
                "merchant_id":           row["merchant_id"],
                "fee_deducted":          max(0.0, fee),
                "settlement_delay_days": delay,
            })
        else:
            unmatched_rows.append(row.to_dict())

    matched_df   = pd.DataFrame(matched_rows)
    unmatched_df = pd.DataFrame(unmatched_rows) if unmatched_rows else pd.DataFrame(
        columns=["transaction_id", "amount", "timestamp", "merchant_id"]
    )
    return matched_df, unmatched_df


def _labels_and_eligibility(ground_truth: pd.DataFrame, transaction_ids: pd.Series) -> tuple:
    """Return anomaly labels and a boolean mask for ML-eligible rows."""
    gt = ground_truth.set_index("transaction_id")
    labels = transaction_ids.map(
        lambda tid: int(gt.at[tid, "bucket"] in ANOMALY_BUCKETS)
    ).to_numpy(dtype=int)

    if "ml_eligible" in gt.columns:
        eligible = transaction_ids.map(gt["ml_eligible"]).fillna(False).astype(bool)
    else:
        eligible = transaction_ids.map(
            lambda tid: gt.at[tid, "bucket"] not in DATABASE_RULE_BUCKETS
        ).astype(bool)
    return labels, eligible


def _estimator_candidates(n_rows: int) -> list:
    """Scale the tree-count search space with the current training population."""
    base = max(1, int(np.ceil(np.sqrt(n_rows))))
    return sorted({base, base * 2, base * 3})


def _fit_and_calibrate(X: np.ndarray, y: np.ndarray, model_name: str) -> tuple:
    """Fit unsupervised candidates and select a precision-first score cutoff."""
    if int(y.sum()) < 2:
        raise ValueError(
            f"{model_name} needs at least two ML-eligible anomaly rows for calibration."
        )

    X_train, X_validation, y_train, y_validation = train_test_split(
        X, y,
        test_size=validation_fraction(y),
        random_state=RANDOM_STATE,
        stratify=y,
    )

    candidates = []
    for n_estimators in _estimator_candidates(len(X_train)):
        candidate = IsolationForest(
            n_estimators=n_estimators,
            contamination="auto",
            random_state=RANDOM_STATE,
        )
        candidate.fit(X_train)
        calibration = select_precision_first_threshold(
            y_validation, -candidate.score_samples(X_validation)
        )
        candidates.append((candidate, calibration, n_estimators))
        print(
            f"[train] {model_name}: n={n_estimators}, "
            f"precision={calibration.precision:.3f}, "
            f"recall={calibration.recall:.3f}, "
            f"alert_rate={calibration.alert_rate:.3f}"
        )

    _, best_calibration, best_n_estimators = sorted(
        candidates,
        key=lambda item: (
            -item[1].precision,
            item[1].alert_rate,
            -item[1].f1,
            item[2],
        ),
    )[0]

    # Refit on all ML-eligible data — labels are used only for cutoff calibration.
    final_model = IsolationForest(
        n_estimators=best_n_estimators,
        contamination="auto",
        random_state=RANDOM_STATE,
    )
    final_model.fit(X)
    final_calibration = select_precision_first_threshold(
        y_validation, -final_model.score_samples(X_validation)
    )
    apply_score_threshold(final_model, final_calibration.score_threshold)

    metadata = {
        "model": model_name,
        "n_estimators": best_n_estimators,
        "validation_fraction": validation_fraction(y),
        **final_calibration.to_dict(),
    }
    return final_model, metadata


def train():
    """Load training CSVs, run waterfall, fit both models, save to MODELS_DIR."""
    print("[train] Loading training data ...")
    ledger       = pd.read_csv(DATA_DIR / "train_ledger.csv",       dtype={"amount": str})
    bank         = pd.read_csv(DATA_DIR / "train_bank.csv",         dtype={"deposit_amount": str})
    ground_truth = pd.read_csv(DATA_DIR / "train_ground_truth.csv")

    print(f"[train] {len(ledger)} ledger rows, {len(bank)} bank rows")

    matched_df, unmatched_df = _simple_waterfall(ledger, bank)
    print(f"[train] {len(matched_df)} matched, {len(unmatched_df)} unmatched")

    calibration_metadata = {}
    for model_name, raw_df, feature_builder in [
        ("matched",   matched_df,   matched_features),
        ("unmatched", unmatched_df, unmatched_features),
    ]:
        if raw_df.empty:
            print(f"[train] WARNING: no {model_name} rows — skipping model")
            continue

        labels, eligible = _labels_and_eligibility(ground_truth, raw_df["transaction_id"])
        eligible_df      = raw_df.loc[eligible].reset_index(drop=True)
        eligible_labels  = labels[eligible.to_numpy()]
        print(
            f"[train] {model_name}: {len(eligible_df)} ML-eligible rows, "
            f"{eligible_labels.sum()} labelled anomalies"
        )

        X           = feature_builder(eligible_df)
        model, meta = _fit_and_calibrate(X, eligible_labels, f"model_{model_name}")
        model_path  = MODELS_DIR / f"model_{model_name}.pkl"
        joblib.dump(model, model_path)
        calibration_metadata[model_name] = meta
        print(
            f"[train] {model_path.name} saved "
            f"(rows={len(X)}, features={X.shape[1]}, "
            f"precision={meta['precision']:.3f}, "
            f"recall_floor={meta['recall_floor']:.3f})"
        )

    calib_path = DATA_DIR / "model_calibration.json"
    with open(calib_path, "w") as fh:
        json.dump(calibration_metadata, fh, indent=2)
    print(f"[train] Calibration metadata written to {calib_path}")
    print("[train] Done.")


print("Training functions defined.")

Training functions defined.


## 7. Evaluation

Loads the test split, runs the waterfall, scores both saved models and reports
Precision, Recall, F1, and PR-AUC. Accuracy is deliberately omitted (misleading at <5% anomaly rate).

In [53]:
def load_ground_truth_labels(gt_df: pd.DataFrame, txn_ids: pd.Series) -> np.ndarray:
    """Map transaction IDs to binary labels: 1 = anomaly, 0 = normal."""
    gt_map = gt_df.set_index("transaction_id")["bucket"].to_dict()
    return np.array([
        1 if gt_map.get(tid, "normal") in ANOMALY_BUCKETS else 0
        for tid in txn_ids
    ])


def ml_eligibility_mask(gt_df: pd.DataFrame, txn_ids: pd.Series) -> pd.Series:
    """Mirror rows removed by deterministic PostgreSQL anomaly queries."""
    gt = gt_df.set_index("transaction_id")
    if "ml_eligible" in gt.columns:
        return txn_ids.map(gt["ml_eligible"]).fillna(False).astype(bool)
    return txn_ids.map(
        lambda tid: gt.at[tid, "bucket"] not in DATABASE_RULE_BUCKETS
    ).astype(bool)


def evaluate():
    """Load test CSVs, score both models, print and save metrics."""
    print("[evaluate] Loading test data ...")
    ledger = pd.read_csv(DATA_DIR / "test_ledger.csv",       dtype={"amount": str})
    bank   = pd.read_csv(DATA_DIR / "test_bank.csv",         dtype={"deposit_amount": str})
    gt     = pd.read_csv(DATA_DIR / "test_ground_truth.csv")

    matched_df, unmatched_df = _simple_waterfall(ledger, bank)

    results = {}

    matched_mask   = ml_eligibility_mask(gt, matched_df["transaction_id"])
    unmatched_mask = ml_eligibility_mask(gt, unmatched_df["transaction_id"])
    matched_df     = matched_df.loc[matched_mask].reset_index(drop=True)
    unmatched_df   = unmatched_df.loc[unmatched_mask].reset_index(drop=True)

    # ── Matched model ─────────────────────────────────────────────────────────
    matched_model_path = MODELS_DIR / "model_matched.pkl"
    if not matched_df.empty and matched_model_path.exists():
        model  = joblib.load(matched_model_path)
        X      = matched_features(matched_df)
        scores = model.score_samples(X)
        preds  = (model.predict(X) == -1).astype(int)
        y_true = load_ground_truth_labels(gt, matched_df["transaction_id"])

        pr_auc = average_precision_score(y_true, -scores) if y_true.sum() > 0 else 0.0
        p, r, f, _ = precision_recall_fscore_support(y_true, preds, average="binary", zero_division=0)

        results["matched"] = {
            "n_samples":        len(X),
            "n_anomalies_true": int(y_true.sum()),
            "n_flagged":        int(preds.sum()),
            "precision":        round(float(p), 4),
            "recall":           round(float(r), 4),
            "f1":               round(float(f), 4),
            "pr_auc":           round(float(pr_auc), 4),
        }
        print(f"\n[evaluate] Matched model:\n  {results['matched']}")
    else:
        print("[evaluate] model_matched.pkl not found or no matched rows — skipping")

    # ── Unmatched model ───────────────────────────────────────────────────────
    unmatched_model_path = MODELS_DIR / "model_unmatched.pkl"
    if not unmatched_df.empty and unmatched_model_path.exists():
        model  = joblib.load(unmatched_model_path)
        X      = unmatched_features(unmatched_df)
        scores = model.score_samples(X)
        preds  = (model.predict(X) == -1).astype(int)
        y_true = load_ground_truth_labels(gt, unmatched_df["transaction_id"])

        pr_auc = average_precision_score(y_true, -scores) if y_true.sum() > 0 else 0.0
        p, r, f, _ = precision_recall_fscore_support(y_true, preds, average="binary", zero_division=0)

        results["unmatched"] = {
            "n_samples":        len(X),
            "n_anomalies_true": int(y_true.sum()),
            "n_flagged":        int(preds.sum()),
            "precision":        round(float(p), 4),
            "recall":           round(float(r), 4),
            "f1":               round(float(f), 4),
            "pr_auc":           round(float(pr_auc), 4),
        }
        print(f"\n[evaluate] Unmatched model:\n  {results['unmatched']}")
    else:
        print("[evaluate] model_unmatched.pkl not found or no unmatched rows — skipping")

    out_path = DATA_DIR / "evaluation_results.json"
    with open(out_path, "w") as fh:
        json.dump(results, fh, indent=2)
    print(f"\n[evaluate] Results written to {out_path}")
    return results


print("Evaluation functions defined.")

Evaluation functions defined.


---
## 8. Run the Pipeline

Everything below executes the functions defined above in order.

### 8a. Generate Data

In [54]:
print("=" * 60)
print("STEP 1 — Data Generation")
print("=" * 60)

print("\n--- Generating Training Data ---")
generate(split="train", seed=42, n_rows=N_TRAIN_ROWS)

print("\n--- Generating Test Data ---")
generate(split="test", seed=999, n_rows=N_TEST_ROWS)

STEP 1 — Data Generation

--- Generating Training Data ---
[generate] Loading PaySim from /kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv ...
[generate] train: 50000 ledger rows, 42499 bank rows
  Dynamic configuration: merchants=223, settlement_window=9 days, matched_anomaly_fee=0.0525, matched_anomaly_delay=22 days
  Bucket breakdown:
bucket
exact         27786
group          7563
date_shift     6008
fee_adj        4119
ambiguous      1588
missing        1527
duplicate       961
anomaly         448
  Written: /kaggle/working/ml/data/train_ledger.csv, /kaggle/working/ml/data/train_bank.csv, /kaggle/working/ml/data/train_ground_truth.csv

--- Generating Test Data ---
[generate] Loading PaySim from /kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv ...
[generate] test: 10000 ledger rows, 8492 bank rows
  Dynamic configuration: merchants=100, settlement_window=3 days, matched_anomaly_fee=0.0500, matched_anomaly_delay=6 days
  Bucket brea

### 8b. Train Models

In [55]:
print("=" * 60)
print("STEP 2 — Model Training")
print("=" * 60)

train()

STEP 2 — Model Training
[train] Loading training data ...
[train] 50000 ledger rows, 42499 bank rows
[train] 39098 matched, 10902 unmatched
[train] matched: 38137 ML-eligible rows, 224 labelled anomalies
[train] model_matched: n=189, precision=0.226, recall=0.933, alert_rate=0.024
[train] model_matched: n=378, precision=0.222, recall=0.933, alert_rate=0.025
[train] model_matched: n=567, precision=0.222, recall=0.933, alert_rate=0.025
[train] model_matched.pkl saved (rows=38137, features=6, precision=0.246, recall_floor=0.933)
[train] unmatched: 9314 ML-eligible rows, 224 labelled anomalies
[train] model_unmatched: n=94, precision=0.636, recall=0.933, alert_rate=0.035
[train] model_unmatched: n=188, precision=0.667, recall=0.933, alert_rate=0.034
[train] model_unmatched: n=282, precision=0.682, recall=1.000, alert_rate=0.035
[train] model_unmatched.pkl saved (rows=9314, features=5, precision=0.636, recall_floor=0.933)
[train] Calibration metadata written to /kaggle/working/ml/data/model

### 8c. Evaluate Models

In [56]:
print("=" * 60)
print("STEP 3 — Model Evaluation")
print("=" * 60)

results = evaluate()

STEP 3 — Model Evaluation
[evaluate] Loading test data ...

[evaluate] Matched model:
  {'n_samples': 7578, 'n_anomalies_true': 100, 'n_flagged': 334, 'precision': 0.1766, 'recall': 0.59, 'f1': 0.2719, 'pr_auc': 0.1383}

[evaluate] Unmatched model:
  {'n_samples': 1895, 'n_anomalies_true': 100, 'n_flagged': 164, 'precision': 0.6098, 'recall': 1.0, 'f1': 0.7576, 'pr_auc': 0.9014}

[evaluate] Results written to /kaggle/working/ml/data/evaluation_results.json


### 8d. Display Full Results

In [57]:
print("\n" + "=" * 60)
print("FINAL EVALUATION RESULTS")
print("=" * 60)
print(json.dumps(results, indent=2))

if results:
    df_results = pd.DataFrame(results).T
    display(df_results)


FINAL EVALUATION RESULTS
{
  "matched": {
    "n_samples": 7578,
    "n_anomalies_true": 100,
    "n_flagged": 334,
    "precision": 0.1766,
    "recall": 0.59,
    "f1": 0.2719,
    "pr_auc": 0.1383
  },
  "unmatched": {
    "n_samples": 1895,
    "n_anomalies_true": 100,
    "n_flagged": 164,
    "precision": 0.6098,
    "recall": 1.0,
    "f1": 0.7576,
    "pr_auc": 0.9014
  }
}


,n_samples,n_anomalies_true,n_flagged,precision,recall,f1,pr_auc
matched,7578.0,100.0,334.0,0.1766,0.59,0.2719,0.1383
unmatched,1895.0,100.0,164.0,0.6098,1.00,0.7576,0.9014


### 8e. Calibration Metadata

In [58]:
calib_path = DATA_DIR / "model_calibration.json"
if calib_path.exists():
    with open(calib_path) as fh:
        calib = json.load(fh)
    print(json.dumps(calib, indent=2))
else:
    print("model_calibration.json not found — training may have been skipped.")

{
  "matched": {
    "model": "model_matched",
    "n_estimators": 189,
    "validation_fraction": 0.06696428571428571,
    "score_threshold": 0.6019816793808254,
    "recall_floor": 0.9333333333333333,
    "precision": 0.2459016393442623,
    "recall": 1.0,
    "f1": 0.3947368421052631,
    "alert_rate": 0.023884103367267033,
    "validation_anomalies": 15,
    "validation_rows": 2554
  },
  "unmatched": {
    "model": "model_unmatched",
    "n_estimators": 282,
    "validation_fraction": 0.06696428571428571,
    "score_threshold": 0.6389942693116916,
    "recall_floor": 0.9333333333333333,
    "precision": 0.6363636363636364,
    "recall": 0.9333333333333333,
    "f1": 0.7567567567567568,
    "alert_rate": 0.035256410256410256,
    "validation_anomalies": 15,
    "validation_rows": 624
  }
}
